# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rohindth-08/FlyRank-_Internship_ML-/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

**Finding 1:** *"Pages older than 1 year lose 30% of their organic traffic on average."*
- **My Methodology Question (Label Origin):** How exactly was "loss" defined in the label? Does the target variable compare Year-over-Year (YOY) traffic to account for seasonality, or is it simply a Month-over-Month (MOM) drop? If it is MOM, the "loss" might just be a seasonal summer dip, meaning the label itself is structurally flawed.

**Finding 2:** *"Our model predicts traffic drops with 85% accuracy."*
- **My Methodology Question (Validation Design):** Does the validation design support this claim? Was the 85% accuracy evaluated on a random split, or a strictly time-aware holdout? If the model was trained and tested on interleaved days within the same month, it leaked temporal trends and the 85% claim is highly inflated.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. My model under an honest split (before/after)

Let's prove why splits matter. I will train my ML-08 Random Forest on the exact same dataset twice:
1. **The Naive Random Split:** A simple 80/20 train/test split. (This allows the model to memorize a client's seasonal trends).
2. **The Honest Grouped Split:** Grouped by `client_hash_id`. (This forces the model to generalize to clients it has never seen before).

In [2]:
import pandas as pd
import numpy as np
import duckdb
import os
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
import warnings
warnings.filterwarnings('ignore')

token = os.environ.get('HF_TOKEN')
conn = duckdb.connect()
conn.execute('INSTALL httpfs; LOAD httpfs;')
if token:
    conn.execute(f"CREATE SECRET hf (TYPE HUGGINGFACE, TOKEN '{token}')")

query = """
WITH feb AS (
  SELECT client_hash_id, content_hash_id, SUM(gsc_impressions) as feb_imps, AVG(gsc_avg_position) as feb_pos
  FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-02/*.parquet'
  GROUP BY client_hash_id, content_hash_id
),
mar AS (
  SELECT content_hash_id, SUM(gsc_impressions) as mar_imps
  FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
  GROUP BY content_hash_id
)
SELECT 
  feb.client_hash_id, feb.content_hash_id, feb.feb_imps, feb.feb_pos, mar.mar_imps,
  d.word_count, d.category_count, date_diff('day', CAST(d.content_updated_date AS DATE), DATE '2026-02-28') as days_since_update_feb
FROM feb JOIN mar ON feb.content_hash_id = mar.content_hash_id
JOIN 'hf://datasets/FlyRank/internship-warehouse/dim_content.parquet' d ON feb.content_hash_id = d.content_hash_id
WHERE feb.feb_imps > 100 AND d.content_updated_date IS NOT NULL
"""
df = conn.execute(query).df()
df['target_decay'] = (df['mar_imps'] < df['feb_imps'] * 0.8).astype(int)

features = ['feb_imps', 'feb_pos', 'word_count', 'category_count', 'days_since_update_feb']

def train_and_eval(df_train, df_test):
    pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('rf', RandomForestClassifier(max_depth=5, n_estimators=100, random_state=42))
    ])
    pipeline.fit(df_train[features], df_train['target_decay'])
    scores = pipeline.predict_proba(df_test[features])[:, 1]
    order = np.argsort(-np.asarray(scores))
    return np.asarray(df_test['target_decay'])[order[:50]].mean()

# 1. Naive Random Split
train_naive, test_naive = train_test_split(df, test_size=0.2, random_state=42)
p50_naive = train_and_eval(train_naive, test_naive)

# 2. Honest Grouped Split
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(df, groups=df['client_hash_id']))
p50_grouped = train_and_eval(df.iloc[train_idx], df.iloc[test_idx])

results = pd.DataFrame({
    'Validation Design': ['Naive Random Split (Leaky)', 'Grouped by Client (Honest)'],
    'Precision@50': [f"{p50_naive:.1%}", f"{p50_grouped:.1%}"]
})
display(results)
print("Observation: The naive split memorizes client traffic patterns, inflating the score. The grouped split reveals the true, harder generalization score.")

,Validation Design,Precision@50
0,Naive Random Split (Leaky),70.0%
1,Grouped by Client (Honest),28.0%


Observation: The naive split memorizes client traffic patterns, inflating the score. The grouped split reveals the true, harder generalization score.


In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Leakage audit

I will perform the **Attack Checklist** from Week 3 on my model to ensure no future information is sneaking into the predictions.

In [4]:
print("--- Leakage Audit Checklist ---")
print("✅ Timeline drawn: ALL features (feb_imps, feb_pos, days_since_update_feb) are strictly calculated using data ending on Feb 28th. The target (mar_imps) strictly occurs after March 1st.")
print("✅ No label-derived columns: If I accidentally included `mar_imps` as a feature, the score would jump to 100%. Let's prove it by intentionally leaking it:")

# Intentional Leakage Test
leaky_features = features + ['mar_imps']
pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('rf', RandomForestClassifier(max_depth=5, n_estimators=100, random_state=42))
])
pipeline.fit(df.iloc[train_idx][leaky_features], df.iloc[train_idx]['target_decay'])
leaky_scores = pipeline.predict_proba(df.iloc[test_idx][leaky_features])[:, 1]
order = np.argsort(-np.asarray(leaky_scores))
leaky_p50 = np.asarray(df.iloc[test_idx]['target_decay'])[order[:50]].mean()

print(f"-> Precision@50 with intentional leak: {leaky_p50:.1%}")
print(f"-> My actual honest model score is much lower ({p50_grouped:.1%}), proving I did not leak the label into my features.")
print("✅ Split grouped by repeating entity: Confirmed (GroupShuffleSplit on client_hash_id).")

--- Leakage Audit Checklist ---
✅ Timeline drawn: ALL features (feb_imps, feb_pos, days_since_update_feb) are strictly calculated using data ending on Feb 28th. The target (mar_imps) strictly occurs after March 1st.
✅ No label-derived columns: If I accidentally included `mar_imps` as a feature, the score would jump to 100%. Let's prove it by intentionally leaking it:


-> Precision@50 with intentional leak: 100.0%
-> My actual honest model score is much lower (28.0%), proving I did not leak the label into my features.
✅ Split grouped by repeating entity: Confirmed (GroupShuffleSplit on client_hash_id).


In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Claim rewrite

**Bold Claim (Before):**
*"My Random Forest model successfully predicts exactly which pages will decay, allowing us to perfectly prioritize our content refresh pipeline and save clients from losing traffic."*

**Honest Claim (After):**
*"The Random Forest model provides a **directional** signal for identifying content at risk of decay. When evaluated on a sealed, client-grouped split, the model demonstrated a **measured** improvement over the baseline rule, offering a valuable **decision-support** tool for prioritizing content refreshes."*

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.